In [1]:
import polars as pl

In [ ]:
# ==========================================
# 1. Define File Paths
# ==========================================
file = r"D:\DEPI\Final_Project\nsduh-2021-2023-ds0001-bndl-data-tsv_v1\NSDUH 2021_2023.parquet"
output_path = r"D:\DEPI\Final_Project\nsduh-2021-2023-ds0001-bndl-data-tsv_v1\NSDUH_Cleaned.parquet"

In [3]:
df = pl.scan_parquet(file)

print(df.collect_schema().names()[:20])
print("عدد الأعمدة:", len(df.collect_schema().names()))

['YEAR', 'AGE3', 'PDEN10', 'NEWRACE2', 'IRINSUR4', 'IRHHSIZ2', 'IRPINC3', 'IRFAMIN3', 'IRKI17_2', 'IRHH65_2', 'WRKSTATWK2', 'EDUSCHGRD2', 'VESTR_C', 'VEREP', 'IREDUHIGHST2', 'IRMARIT', 'WRKDPSTWK', 'WRKHADJOB', 'WRK35WKUS', 'WRKRSNNOT']
عدد الأعمدة: 2639


In [ ]:
# ==========================================
# 2. Optimized Feature Selection (Star Schema Ready)
# ==========================================

columns_to_keep = [ 
    "ANALWT2_C1",    # Survey weight (sample weight)
    "YEAR",         # Survey year
    "AGE3",         # Respondent's age
    "CATAGE",       # Age category
    "IRSEX",        # Respondent's sex
    "NEWRACE2",     # Race/Ethnicity
    "EDUHIGHCAT",   # Highest education level
    "IRWRKSTAT",     # Employment status
    "IRPINC3",      # Personal income category
    "PDEN10",       # Population density / Urbanicity
    "POVERTY3",     # Poverty status
    "HEALTH2",      # Self-rated overall health
    "AMDEYR",       # Major depressive episode in the past year
    "KSSLR6YR",     # Psychological distress (Kessler-6 score)
    "IRSUICTHNK",   # Serious suicidal thoughts in the past year
    "ALCYR",        # Alcohol use in the past year
    "CIGYR",        # Cigarette use in the past year
    "MRJYR",         # Marijuana use in the past year
    "PYUD5ALC",     # DSM-5 Alcohol Use Disorder
    "PYUD5MRJ",     # DSM-5 Marijuana Use Disorder
    "MHTNSEEKPY"    # Sought mental health treatment in the past year
]
print("Starting the NSDUH data processing pipeline...")

Starting the NSDUH data processing pipeline...


In [5]:
columns = df.collect_schema().names()

missing = [c for c in columns_to_keep if c not in columns]

print(missing)

['STATEFIP']


In [6]:
# ==========================================
# 3. Lazy Loading & Selection
# ==========================================
print("Loading Parquet and selecting columns...")
df_lazy = pl.scan_parquet(file).select(columns_to_keep)


Loading Parquet and selecting columns...


In [ ]:
# ==========================================
# 4. Preprocessing (-9 to Null)
# ==========================================

print("Handling missing values (-9 to Null)...")
cleaned_lazy_df = df_lazy.with_columns(
    pl.all().map_batches(lambda s: s.replace(-9, None))
)

Handling missing values (-9 to Null)...


In [8]:
negative_values = {}

for col in columns_to_keep:
    vals = (
        df_lazy
        .select(
            pl.col(col)
            .filter(pl.col(col) < 0)
            .unique()
            .sort()
        )
        .collect()
        .to_series()
        .to_list()
    )

    if vals:
        negative_values[col] = vals

print(negative_values)

{'MHTNSEEKPY': [-9.0]}


In [ ]:
# ==========================================
# 5. Execution & Saving
# ==========================================
print("Executing and saving to Parquet (Star Schema optimized)...")

cleaned_lazy_df.sink_parquet(output_path)

print(f"Pipeline executed successfully!")
print(f"File saved at: {output_path}")

Executing and saving to Parquet (Star Schema optimized)...
Pipeline executed successfully!
File saved at: D:\DEPI\Final_Project\nsduh-2021-2023-ds0001-bndl-data-tsv_v1\NSDUH_Cleaned.parquet
